# 03 · Survival analysis (starter): how long until a matching flat comes up?
**Owner:** Pricing & Availability Lead · **Needs:** step 1b done; `lifelines` (Terminal → Run Task → *Install modelling libraries*)

**Question.** For a given block and flat type, how many months pass between one resale and the next?
That tells a buyer how readily flats like the one they want come onto the market.

| Design choice | Decision | Why |
|---|---|---|
| Unit | block × flat type (e.g. 4-room flats in Blk 406 AMK Ave 10) | The resale data has no unit numbers, but block + flat type is consistent |
| Event | a resale of that flat type in that block | Directly observed in HDB data (no scraping needed) |
| Duration | months between successive sales; same-month repeat = 0.5 months | Data is monthly |
| Right-censoring | the last spell runs to Sep 2026 with no sale yet | Standard; KM and Cox handle it |
| Never-sold groups | **included** as censored spells | Leaving them out would make waits look far shorter than they are |
| Before the first sale (old blocks) | dropped | The previous sale happened before 2017, so the spell's start is unknown |
| Newer blocks | clock starts when they become eligible (completion year + 5 for MOP) | Their first-ever resale wait is observable |

Location covariates (MRT, amenities) are added in section 7 once `block_features.csv` exists (after tonight's geocoding).

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, "../src")
import config as C
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test
from lifelines.utils import concordance_index

TYPES = ["2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE"]
COL = dict(zip(TYPES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e6e3", "axes.axisbelow": True, "legend.frameon": False})

def month_index(m):                       # "2017-01" -> integer months, so gaps are simple subtraction
    return m.str[:4].astype(int) * 12 + m.str[5:7].astype(int) - 1

df = pd.read_csv(C.RESALE_CLEAN, dtype={"block": str})
df = df[df["in_scope"] & df["flat_type"].isin(TYPES)].copy()
df["t"] = month_index(df["month"])
T0, T_END = df["t"].min(), df["t"].max()                 # Jan 2017 .. last complete month
print(f"{len(df):,} sales, window {df['month'].min()} to {df['month'].max()} ({T_END - T0 + 1} months)")

## 1 · Exposure: how many flats of each type are in each block
From HDB Property Information. A block with 200 four-room flats should see more sales than one with 20, so this is the key control.

In [ ]:
p = pd.read_csv(C.PROPINFO, dtype={"blk_no": str})
p = p[p["residential"] == "Y"].copy()
p["address"] = (p["blk_no"].str.strip().str.upper() + " "
                + p["street"].str.strip().str.upper().str.replace(r"\s+", " ", regex=True))
cols = {"2room_sold": "2 ROOM", "3room_sold": "3 ROOM", "4room_sold": "4 ROOM", "5room_sold": "5 ROOM", "exec_sold": "EXECUTIVE"}
stock = (p.melt(id_vars=["address", "year_completed", "max_floor_lvl", "bldg_contract_town"],
                value_vars=list(cols), var_name="col", value_name="units")
           .query("units > 0").assign(flat_type=lambda d: d["col"].map(cols)).drop(columns="col"))
# A block becomes resellable after the 5-year MOP; we only know the completion year, so assume mid-year
stock["t_eligible"] = (stock["year_completed"] + 5) * 12 + 6
print(f"{len(stock):,} block x flat-type groups with sold units")

## 2 · Build the spells
One row per waiting spell: `duration` (months), `event` (1 = ended in a sale, 0 = still waiting at Sep 2026).

In [ ]:
def build_spells(sales, stock, t0, t_end):
    rows = []
    sales_by_group = sales.groupby(["address", "flat_type"])["t"].apply(lambda s: np.sort(s.to_numpy()))
    for g in stock.itertuples():
        times = sales_by_group.get((g.address, g.flat_type), np.array([], dtype=int))
        old = g.t_eligible <= t0                                   # resellable before the data starts
        start = None if old else g.t_eligible                      # new blocks: clock starts at eligibility
        if old and len(times) == 0:
            start = t0                                             # never sold in the whole window
        for t in times:
            if start is not None and t >= start:
                rows.append((g.address, g.flat_type, start, max(t - start, 0.5), 1))
            start = t                                              # next spell starts at this sale
        if start is not None and start <= t_end:
            rows.append((g.address, g.flat_type, start, max(t_end - start, 0.5), 0))   # still waiting
    return pd.DataFrame(rows, columns=["address", "flat_type", "t_start", "duration", "event"])

spells = build_spells(df, stock, T0, T_END)
spells = spells.merge(stock[["address", "flat_type", "units", "year_completed", "max_floor_lvl", "bldg_contract_town"]],
                      on=["address", "flat_type"], how="left")
spells["start_year"] = spells["t_start"] // 12
spells.to_csv(C.PROCESSED / "survival_spells.csv", index=False)
print(f"{len(spells):,} spells · {spells['event'].mean():.1%} end in a sale · "
      f"{(spells['event'] == 0).sum():,} censored -> data/processed/survival_spells.csv")

## 3 · Sanity checks

In [ ]:
print(spells.groupby("flat_type").agg(spells=("event", "size"), events=("event", "sum"),
                                       median_duration=("duration", "median")).loc[TYPES].to_string())
missing = set(map(tuple, df[["address", "flat_type"]].drop_duplicates().values)) - set(map(tuple, stock[["address", "flat_type"]].values))
print(f"\nGroups with sales but missing from HDB Property Information: {len(missing)} (dropped)")
never = spells.groupby(["address", "flat_type"])["event"].sum().eq(0).sum()
print(f"Groups that never sold in the window (fully censored): {never:,}")

**Finding:** _spell counts, censoring share, anything odd._

## 4 · Kaplan–Meier: waiting-time curves by flat type
The curve shows the share of blocks still waiting for their next sale after *t* months.

In [ ]:
fig, ax = plt.subplots()
kmf, medians = KaplanMeierFitter(), {}
for t in TYPES:
    d = spells[spells["flat_type"] == t]
    kmf.fit(d["duration"], d["event"], label=t.title())
    kmf.plot_survival_function(ax=ax, ci_show=False, color=COL[t], linewidth=1.8)
    medians[t] = kmf.median_survival_time_
ax.set_xlim(0, 60); ax.set_xlabel("Months since last sale"); ax.set_ylabel("Share still waiting")
ax.set_title("Kaplan–Meier: time until the next sale in the same block and flat type", loc="left")
plt.show()
print("Median months to next sale:", {k.title(): round(v, 1) for k, v in medians.items()})
res = multivariate_logrank_test(spells["duration"], spells["flat_type"], spells["event"])
print(f"Log-rank test across flat types: chi2 = {res.test_statistic:,.0f}, p = {res.p_value:.2g}")

**Finding:** _which flat types turn over fastest; are the differences significant?_

## 5 · Exposure matters: KM by number of units of that type in the block

In [ ]:
spells["units_band"] = pd.cut(spells["units"], [0, 20, 50, 100, 1e4], labels=["1-20", "21-50", "51-100", "100+"])
fig, ax = plt.subplots()
for band, c in zip(["1-20", "21-50", "51-100", "100+"], ["#cde2fb", "#86b6ef", "#2a78d6", "#184f95"]):
    d = spells[spells["units_band"] == band]
    kmf.fit(d["duration"], d["event"], label=f"{band} units")
    kmf.plot_survival_function(ax=ax, ci_show=False, color=c, linewidth=1.8)
ax.set_xlim(0, 60); ax.set_xlabel("Months since last sale"); ax.set_ylabel("Share still waiting")
ax.set_title("More flats of a type in the block, shorter waits", loc="left"); plt.show()

**Finding:** _how strongly block size drives waits. This is why `log(units)` must be in the Cox model._

## 6 · Cox proportional hazards (starter, no location yet)
Hazard ratio > 1 = sales happen faster. Covariates available now: flat type, region, log units (exposure), block age at spell
start, block height, and the market period (a 'from 2020' indicator for the boom years). Standard errors are clustered by block,
because spells from the same block are not independent.

Validation, as in the proposal: fit on spells **starting before 2023**, then score the C-index on spells starting 2023+.

_Runtime: about 5–6 minutes (clustered standard errors on ~146k spells). Let it run._

In [ ]:
region_of = {t: r for r, ts in C.REGIONS.items() for t in ts}
# Never-sold blocks have no resale rows, so take town from HDB's building-contract-town code.
# The code -> town mapping is learned from blocks that appear in both datasets.
known = stock[["address", "bldg_contract_town"]].drop_duplicates().merge(df[["address", "town"]].drop_duplicates(), on="address")
code_town = known.groupby("bldg_contract_town")["town"].agg(lambda s: s.mode()[0])
X = spells.copy()
X["region"] = X["bldg_contract_town"].map(code_town).map(region_of)
print(f"Region found for {X['region'].notna().mean():.2%} of spells; the rest are dropped from the Cox model")
X = X[X["region"].notna()]
X["log_units"] = np.log(X["units"])
X["block_age"] = X["start_year"] - X["year_completed"]
# Market period: one 'from 2020' indicator (the boom). Finer periods can't be validated on a time split,
# because the test years would get a period the model never saw.
X["from_2020"] = (X["start_year"] >= 2020).astype(float)
design = pd.get_dummies(X[["flat_type", "region"]], drop_first=True, dtype=float)
design[["log_units", "block_age", "max_floor_lvl", "from_2020"]] = X[["log_units", "block_age", "max_floor_lvl", "from_2020"]]
design[["duration", "event", "address"]] = X[["duration", "event", "address"]]
design = design.dropna()

train, test = design[X.loc[design.index, "start_year"] < 2023], design[X.loc[design.index, "start_year"] >= 2023]
cph = CoxPHFitter(penalizer=0.01)
cph.fit(train, duration_col="duration", event_col="event", cluster_col="address")
cph.print_summary(decimals=3, columns=["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"])
c_test = concordance_index(test["duration"], -cph.predict_partial_hazard(test), test["event"])
print(f"\nC-index, train (spells starting before 2023): {cph.concordance_index_:.3f} · test (2023+): {c_test:.3f}")

**Finding:** _which factors speed up or slow down sales; does the C-index hold up on the later period?_

In [ ]:
# Proportional-hazards check on a sample (the full data makes every small deviation 'significant')
cph_s = CoxPHFitter(penalizer=0.01).fit(train.sample(20000, random_state=0).drop(columns="address"),
                                        duration_col="duration", event_col="event")
cph_s.check_assumptions(train.sample(20000, random_state=0).drop(columns="address"), p_value_threshold=0.01, show_plots=False)

**Finding:** _which covariates violate proportional hazards, and the remedy (stratify on them, e.g. `strata=['from_2020']`, or add a time interaction)._

## 7 · Next, after tonight's processed data
1. **Add location covariates**: merge `data/processed/block_features.csv` on `address` (MRT distance, hawker, schools, amenities).
   Fully unsold blocks were never geocoded (they have no resale rows); re-run geocoding for them, or note the gap.
2. **Turn block-level hazards into a profile-level wait.** A buyer wants *any* flat matching their profile, not a specific block.
   If block *b* has monthly sale rate λ_b (from the Cox model), sales in a profile arrive at rate Σλ_b across its blocks, so the
   expected wait for the next one is about 1 / Σλ_b.
3. **Write `outputs/expected_wait.csv`** (profile_id, wait_months) to replace the placeholder, in time for the 16 Oct check-in.